## 2.1 理论计算题（卷积输出尺寸与计算量）

输入一张大小为 $3 \times 32 \times 32$（通道数 $\times$ 高 $\times$ 宽）的彩色图像。通过一个卷积层，该层包含 $16$ 个卷积核，每个卷积核的大小为 $3 \times 5 \times 5$。设定填充（Padding）为 $2$，步幅（Stride）为 $2$。

### 1. 输出特征图尺寸

输出通道数 = 卷积核个数 = $16$

输出高度：
$$H_{out} = \left\lfloor \frac{H_{in} + 2P - K_H}{S} \right\rfloor + 1 = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16$$

输出宽度：
$$W_{out} = \left\lfloor \frac{W_{in} + 2P - K_W}{S} \right\rfloor + 1 = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = 16$$

**输出特征图尺寸：$16 \times 16 \times 16$（通道数 $\times$ 高 $\times$ 宽）**

### 2. 单个输出通道一个像素值的点乘次数

每个卷积核大小为 $3 \times 5 \times 5$，计算一个像素需要与卷积核做一次完整的点积运算：

$$\text{乘法次数} = C_{in} \times K_H \times K_W = 3 \times 5 \times 5 = 75$$

**单个输出通道的一个像素值需要对输入进行 $\mathbf{75}$ 次点乘操作。**

全图总乘法量：$16 \times 16 \times 16 \times 75 = 307,200$ 次（可用于验证）。

In [ ]:
import numpy as np

def max_pool2d(x, kernel_size, stride=None, padding=0):
    '''
    手动实现支持 stride 和 padding 的二维最大池化前向传播。

    参数:
        x:           输入张量, 形状 (N, C, H, W)
        kernel_size: 池化窗口大小 (int)
        stride:      步幅 (int), 默认等于 kernel_size
        padding:     填充大小 (int), 默认 0

    返回:
        池化后的张量
    '''
    if stride is None:
        stride = kernel_size

    N, C, H, W = x.shape

    # 计算输出尺寸
    H_out = (H + 2 * padding - kernel_size) // stride + 1
    W_out = (W + 2 * padding - kernel_size) // stride + 1

    # 对输入做 padding（在 H 和 W 维度两侧填充 0）
    if padding > 0:
        x_padded = np.pad(x, ((0, 0), (0, 0),
                               (padding, padding), (padding, padding)),
                          mode='constant', constant_values=-np.inf)
    else:
        x_padded = x

    # 初始化输出
    out = np.zeros((N, C, H_out, W_out))

    # 滑动窗口进行最大池化
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            h_end = h_start + kernel_size
            w_start = j * stride
            w_end = w_start + kernel_size

            # 取出窗口并取最大值
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))

    return out


# ====== 测试 ======
# 构造输入: N=2, C=2, H=6, W=6
x_test = np.array([
    [[[1, 2, 3, 4, 5, 6],
      [7, 8, 9, 0, 1, 2],
      [3, 4, 5, 6, 7, 8],
      [9, 0, 1, 2, 3, 4],
      [5, 6, 7, 8, 9, 0],
      [1, 2, 3, 4, 5, 6]],

     [[6, 5, 4, 3, 2, 1],
      [0, 9, 8, 7, 6, 5],
      [4, 3, 2, 1, 0, 9],
      [8, 7, 6, 5, 4, 3],
      [2, 1, 0, 9, 8, 7],
      [6, 5, 4, 3, 2, 1]]],

    [[[0, 1, 2, 3, 4, 5],
      [5, 4, 3, 2, 1, 0],
      [9, 8, 7, 6, 5, 4],
      [3, 2, 1, 0, 9, 8],
      [7, 6, 5, 4, 3, 2],
      [1, 0, 9, 8, 7, 6]],

     [[9, 8, 7, 6, 5, 4],
      [3, 2, 1, 0, 9, 8],
      [7, 6, 5, 4, 3, 2],
      [1, 0, 9, 8, 7, 6],
      [5, 4, 3, 2, 1, 0],
      [9, 8, 7, 6, 5, 4]]]
], dtype=np.float64)

print("===== 输入 x 形状:", x_test.shape, "=====")
print("x[0, 0]:\n", x_test[0, 0])
print("\nx[0, 1]:\n", x_test[0, 1])

# 测试: kernel_size=2, stride=2, padding=0
out1 = max_pool2d(x_test, kernel_size=2, stride=2, padding=0)
print("\n===== kernel_size=2, stride=2, padding=0 =====")
print("输出形状:", out1.shape)
print("out[0, 0]:\n", out1[0, 0])
print("out[0, 1]:\n", out1[0, 1])

# 测试: kernel_size=3, stride=2, padding=1
out2 = max_pool2d(x_test, kernel_size=3, stride=2, padding=1)
print("\n===== kernel_size=3, stride=2, padding=1 =====")
print("输出形状:", out2.shape)
print("out[0, 0]:\n", out2[0, 0])

# 与 PyTorch 对比验证
import torch
x_torch = torch.tensor(x_test)
pool = torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
out_torch = pool(x_torch)
print("\n===== PyTorch 验证 (kernel_size=2, stride=2, padding=0) =====")
print("PyTorch out[0, 0]:\n", out_torch[0, 0].numpy())
print("手动实现与 PyTorch 一致:", np.allclose(out1, out_torch.numpy()))

===== 输入 x 形状: (2, 2, 6, 6) =====
x[0, 0]:
 [[1. 2. 3. 4. 5. 6.]
 [7. 8. 9. 0. 1. 2.]
 [3. 4. 5. 6. 7. 8.]
 [9. 0. 1. 2. 3. 4.]
 [5. 6. 7. 8. 9. 0.]
 [1. 2. 3. 4. 5. 6.]]

x[0, 1]:
 [[6. 5. 4. 3. 2. 1.]
 [0. 9. 8. 7. 6. 5.]
 [4. 3. 2. 1. 0. 9.]
 [8. 7. 6. 5. 4. 3.]
 [2. 1. 0. 9. 8. 7.]
 [6. 5. 4. 3. 2. 1.]]

===== kernel_size=2, stride=2, padding=0 =====
输出形状: (2, 2, 3, 3)
out[0, 0]:
 [[8. 9. 6.]
 [9. 6. 8.]
 [6. 8. 9.]]
out[0, 1]:
 [[9. 8. 6.]
 [8. 6. 9.]
 [6. 9. 8.]]

===== kernel_size=3, stride=2, padding=1 =====
输出形状: (2, 2, 3, 3)
out[0, 0]:
 [[8. 9. 6.]
 [9. 9. 8.]
 [9. 8. 9.]]

===== PyTorch 验证 (kernel_size=2, stride=2, padding=0) =====
PyTorch out[0, 0]:
 [[8. 9. 6.]
 [9. 6. 8.]
 [6. 8. 9.]]
手动实现与 PyTorch 一致: True


## 3.1 理论计算题（VGG 卷积核参数量对比）

在 VGG 网络中，作者频繁使用多个 $3 \times 3$ 卷积核级联来代替较大的卷积核（如 $5 \times 5$ 或 $7 \times 7$）。假设输入和输出的特征图通道数均为 $C$。

### 1. 一个 $5 \times 5$ 卷积层（不带偏置）的参数量

一个卷积核的大小为 $C \times 5 \times 5$，共有 $C$ 个输出通道即 $C$ 个卷积核：

$$\text{参数量}_{5\times5} = C \times C \times 5 \times 5 = 25C^2$$

### 2. 两个串联的 $3 \times 3$ 卷积层（不带偏置）的总参数量

第一层：$C$ 个输入通道，$C$ 个输出通道
$$\text{参数}_{\text{layer1}} = C \times C \times 3 \times 3 = 9C^2$$

第二层：$C$ 个输入通道，$C$ 个输出通道
$$\text{参数}_{\text{layer2}} = C \times C \times 3 \times 3 = 9C^2$$

总参数量：
$$\text{参数量}_{3\times3 \text{ 串联}} = 9C^2 + 9C^2 = 18C^2$$

### 对比分析

$$\frac{\text{参数量}_{3\times3 \text{ 串联}}}{\text{参数量}_{5\times5}} = \frac{18C^2}{25C^2} = \frac{18}{25} = 0.72$$

两个 $3 \times 3$ 卷积层串联的参数量仅为单个 $5 \times 5$ 卷积层的 **72%**，而其感受野与一个 $5 \times 5$ 卷积核相同（均为 $5 \times 5$），同时拥有更多的非线性激活函数，表达能力更强。这就是 VGG 用小卷积核级联替代大卷积核的优势。

In [ ]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    '''
    NiN 块：由 1 个普通卷积层 + 2 个 1x1 卷积层级联组成，
    每层卷积后紧跟 ReLU 激活层。
    '''
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size,
                       stride=stride, padding=padding),
            nn.ReLU(),
            # 第一个 1x1 卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个 1x1 卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)


# ====== 测试 ======
# 构造输入: N=2, C=3, H=32, W=32
x = torch.randn(2, 3, 32, 32)
print("输入形状:", x.shape)

# 创建 NiN 块
nin = NiNBlock(in_channels=3, out_channels=64, kernel_size=5, stride=1, padding=2)
out = nin(x)
print("输出形状:", out.shape)

# 统计参数量
total_params = sum(p.numel() for p in nin.parameters())
print(f"总参数量: {total_params:,}")

# 打印各层参数
print("\n各层结构:")
print(nin)

输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 64, 32, 32])
总参数量: 13,184

各层结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)


## 4.1 理论计算题（Batch Normalization）

在一个小批量训练中，某一通道内某一特定空间位置的特征值在 $4$ 个样本上的输出分别为：
$$x_1 = 2, \quad x_2 = 4, \quad x_3 = 6, \quad x_4 = 8$$

缩放参数 $\gamma = 2$，平移参数 $\beta = 1$，常数 $\epsilon = 0$。

### 步骤 1：计算均值 $\mu$

$$\mu = \frac{1}{4}(2 + 4 + 6 + 8) = \frac{20}{4} = 5$$

### 步骤 2：计算方差 $\sigma^2$

$$\sigma^2 = \frac{1}{4}\left[(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2\right] = \frac{9 + 1 + 1 + 9}{4} = \frac{20}{4} = 5$$

### 步骤 3：标准化

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}$$

$$\hat{x}_1 = \frac{-3}{\sqrt{5}},\quad \hat{x}_2 = \frac{-1}{\sqrt{5}},\quad \hat{x}_3 = \frac{1}{\sqrt{5}},\quad \hat{x}_4 = \frac{3}{\sqrt{5}}$$

### 步骤 4：缩放和平移

$$y_i = \gamma \cdot \hat{x}_i + \beta = 2 \cdot \hat{x}_i + 1$$

**最终结果（精确值 + 近似值）：**

| 样本 | 精确值 | 近似值 |
|------|--------|--------|
| $y_1$ | $1 - \frac{6}{\sqrt{5}}$ | $\approx -1.6833$ |
| $y_2$ | $1 - \frac{2}{\sqrt{5}}$ | $\approx 0.1056$ |
| $y_3$ | $1 + \frac{2}{\sqrt{5}}$ | $\approx 1.8944$ |
| $y_4$ | $1 + \frac{6}{\sqrt{5}}$ | $\approx 3.6833$ |

In [ ]:
import torch
import torch.nn as nn


class Residual(nn.Module):
    '''
    ResNet 残差块：
    - 两个 3x3 卷积层（相同输出通道数），每层后跟 BatchNorm
    - 如果 use_1x1conv=True，使用 1x1 卷积调整输入的通道数和形状
    - 跳跃连接：f(x) + x，然后 ReLU
    '''
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                    stride=stride)
        else:
            self.conv3 = None

        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x if self.conv3 is None else self.conv3(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)


# ====== 测试 1: use_1x1conv=False ======
print("===== use_1x1conv=False =====")
x1 = torch.randn(2, 3, 32, 32)
block1 = Residual(in_channels=3, out_channels=3, use_1x1conv=False)
out1 = block1(x1)
print(f"输入形状: {x1.shape}")
print(f"输出形状: {out1.shape}")
print(f"输入输出形状一致: {x1.shape == out1.shape}")

# ====== 测试 2: use_1x1conv=True（通道数不同） ======
print("\n===== use_1x1conv=True（改变通道数）=====")
x2 = torch.randn(2, 3, 32, 32)
block2 = Residual(in_channels=3, out_channels=64, use_1x1conv=True)
out2 = block2(x2)
print(f"输入形状: {x2.shape}")
print(f"输出形状: {out2.shape}")
print(f"参数量: {sum(p.numel() for p in block2.parameters()):,}")

# ====== 测试 3: use_1x1conv=True + stride=2（下采样） ======
print("\n===== use_1x1conv=True + stride=2（下采样）=====")
x3 = torch.randn(2, 3, 32, 32)
block3 = Residual(in_channels=3, out_channels=64, use_1x1conv=True, stride=2)
out3 = block3(x3)
print(f"输入形状: {x3.shape}")
print(f"输出形状: {out3.shape}")
print(f"参数量: {sum(p.numel() for p in block3.parameters()):,}")

===== use_1x1conv=False =====
输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 3, 32, 32])
输入输出形状一致: True

===== use_1x1conv=True（改变通道数）=====
输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 64, 32, 32])
参数量: 39,232

===== use_1x1conv=True + stride=2（下采样）=====
输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 64, 16, 16])
参数量: 39,232


## 5.1 理论计算题（微调 Fine-tuning）

### 1. 为什么对底层特征提取层设置较小学习率，而对顶层输出层设置较大学习率？

在大型源数据集（如 ImageNet）上预训练的模型，其**底层特征提取层**已经学到了非常通用的视觉特征（如边缘、纹理、形状等），这些特征对不同视觉任务具有良好的迁移性。如果对底层设置较大的学习率，会破坏这些已经学好的通用特征，导致"灾难性遗忘"（Catastrophic Forgetting）。因此，对底层使用小学习率（甚至冻结参数）是为了**保留预训练模型学到的通用知识**。

而**顶层输出层**是随机初始化、专门为目标任务设计的，需要从零开始学习。使用较大学习率可以让顶层快速收敛，适应新任务的类别空间。

简而言之：**底层学的是通用特征（保留），顶层学的是任务特定特征（快速学习）**。

### 2. 目标数据集非常小且与源数据集非常相似时，应该采取什么微调策略？

当目标数据集非常小且与源数据集高度相似时，应采取**"冻结特征提取 + 仅训练分类头"**的策略：

1. **冻结所有预训练的卷积层**（设为不可训练），将其作为固定的特征提取器；
2. **只训练新初始化的全连接分类层**（输出层）；

原因分析：
- 数据集太小 → 如果解冻过多参数，模型容易**过拟合**
- 数据与源数据相似 → 源数据上学到的特征已经足够好，**无需大幅调整**底层特征
- 冻结底层还可以大幅降低训练参数量和计算开销，训练更快更稳定

具体步骤：去掉原模型最后的全连接层 → 添加适应目标任务的新全连接层 → 冻结除新层外的所有参数 → 使用较小的学习率训练新层。

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# 构建图像增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪（面积比例 0.08~1.0）并缩放到 224x224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),

    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),

    # 3. 随机改变亮度、对比度、饱和度（变化范围 0.5）
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),

    # 4. 转换为 PyTorch 张量
    transforms.ToTensor(),
])

print("===== 图像增广管道 =====")
print(augmentation_pipeline)
print()

# ====== 测试：使用随机图像验证管道 ======
# 生成一张随机 RGB 图像 (H=480, W=640, C=3)
random_img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
img_pil = Image.fromarray(random_img)

print(f"原始图像大小: {img_pil.size}")

# 应用增广 3 次，观察效果
for i in range(3):
    augmented = augmentation_pipeline(img_pil)
    print(f"第 {i+1} 次增广 -> 张量形状: {augmented.shape}, "
          f"值域范围: [{augmented.min().item():.3f}, {augmented.max().item():.3f}]")

# 验证：使用真实场景
print("\n===== 说明 =====")
print("1. RandomResizedCrop: 随机裁剪后缩放到 224x224")
print("2. RandomHorizontalFlip: 50% 概率水平翻转")
print("3. ColorJitter: 亮度/对比度/饱和度随机抖动 ±50%")
print("4. ToTensor: 转换为 [0, 1] 范围的张量 (C, H, W)")

===== 图像增广管道 =====
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

原始图像大小: (640, 480)
第 1 次增广 -> 张量形状: torch.Size([3, 224, 224]), 值域范围: [0.000, 1.000]
第 2 次增广 -> 张量形状: torch.Size([3, 224, 224]), 值域范围: [0.000, 0.675]
第 3 次增广 -> 张量形状: torch.Size([3, 224, 224]), 值域范围: [0.000, 0.682]

===== 说明 =====
1. RandomResizedCrop: 随机裁剪后缩放到 224x224
2. RandomHorizontalFlip: 50% 概率水平翻转
3. ColorJitter: 亮度/对比度/饱和度随机抖动 ±50%
4. ToTensor: 转换为 [0, 1] 范围的张量 (C, H, W)


## 6.1 理论计算题（IoU 计算）

已知真实框（Ground Truth）$A = [10, 10, 50, 50]$，预测框（Prediction Box）$B = [30, 30, 70, 70]$。

格式说明：$[x_{\min}, y_{\min}, x_{\max}, y_{\max}]$（左上角和右下角坐标）。

### 1. 计算交集（Intersection）

交集矩形的坐标取两框的"最大值左上"和"最小值右下"：

$$x_{\min}^{\cap} = \max(10, 30) = 30$$
$$y_{\min}^{\cap} = \max(10, 30) = 30$$
$$x_{\max}^{\cap} = \min(50, 70) = 50$$
$$y_{\max}^{\cap} = \min(50, 70) = 50$$

交集面积：
$$S_{\cap} = (50 - 30) \times (50 - 30) = 20 \times 20 = 400$$

### 2. 计算各框面积

$$S_A = (50 - 10) \times (50 - 10) = 40 \times 40 = 1600$$
$$S_B = (70 - 30) \times (70 - 30) = 40 \times 40 = 1600$$

### 3. 计算并集与 IoU

$$S_{\cup} = S_A + S_B - S_{\cap} = 1600 + 1600 - 400 = 2800$$

$$\text{IoU} = \frac{S_{\cap}}{S_{\cup}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$

**结果：$\text{IoU} = \frac{1}{7} \approx 14.29\%$**

In [ ]:
import torch
import torch.nn.functional as F


def label_smoothing_cross_entropy(logits, target, epsilon=0.1):
    '''
    计算标签平滑后的交叉熵损失。

    标准交叉熵使用独热编码（One-hot），标签平滑将其"软化"：
    - 正确类别的目标概率从 1 变为 1 - epsilon
    - 错误类别的目标概率从 0 变为 epsilon / (K - 1)

    参数:
        logits:  模型原始输出 (N, K)，未经 softmax 处理
        target:  真实标签 (N,)，整数索引
        epsilon: 平滑因子，默认 0.1

    返回:
        标量损失值
    '''
    N, K = logits.shape

    # 构造平滑后的目标分布
    # 所有位置初始化为 epsilon / (K - 1)
    smooth_target = torch.full((N, K), epsilon / (K - 1),
                                device=logits.device, dtype=logits.dtype)
    # 正确类别位置填入 1 - epsilon
    smooth_target.scatter_(1, target.unsqueeze(1), 1 - epsilon)

    # 交叉熵 = -sum(p_i * log(q_i))，其中 p 是平滑标签，q 是 softmax 概率
    log_probs = F.log_softmax(logits, dim=1)
    loss = -(smooth_target * log_probs).sum(dim=1).mean()

    return loss


# ====== 测试 ======
torch.manual_seed(42)

# 模拟数据: 10 个样本，5 分类
N, K = 10, 5
logits = torch.randn(N, K)          # 随机初始化的 logits
target = torch.randint(0, K, (N,))   # 随机真实标签

print("===== 标签平滑交叉熵损失测试 =====")
print(f"样本数 N={N}, 类别数 K={K}")
print(f"平滑因子 epsilon=0.1")
print()

# 标签平滑损失
loss_smooth = label_smoothing_cross_entropy(logits, target, epsilon=0.1)
print(f"标签平滑交叉熵损失 (epsilon=0.1):  {loss_smooth.item():.6f}")

# 标准交叉熵损失（对比）
loss_standard = F.cross_entropy(logits, target)
print(f"标准交叉熵损失:                      {loss_standard.item():.6f}")

# 验证：epsilon=0 时等价于标准交叉熵
loss_eps0 = label_smoothing_cross_entropy(logits, target, epsilon=0.0)
print(f"标签平滑损失 (epsilon=0.0):          {loss_eps0.item():.6f}")
print(f"epsilon=0 时与标准交叉熵一致:          {torch.allclose(loss_eps0, loss_standard)}")

print()
print("===== 标签平滑原理验证 =====")
# 对单个样本手动验证
idx = 0
true_class = target[idx].item()
print(f"样本 0: logits = {logits[idx]}")
print(f"       真实类别 = {true_class}")
print(f"       平滑标签: 正确类 {true_class} -> {1 - 0.1:.1f}, "
      f"其他类别 -> {0.1 / (K-1):.4f}")
probs = F.softmax(logits[idx], dim=0)
print(f"       模型预测概率: {probs}")

===== 标签平滑交叉熵损失测试 =====
样本数 N=10, 类别数 K=5
平滑因子 epsilon=0.1

标签平滑交叉熵损失 (epsilon=0.1):  2.189137
标准交叉熵损失:                      2.221237
标签平滑损失 (epsilon=0.0):          2.221237
epsilon=0 时与标准交叉熵一致:          True

===== 标签平滑原理验证 =====
样本 0: logits = tensor([ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784])
       真实类别 = 3
       平滑标签: 正确类 3 -> 0.9, 其他类别 -> 0.0250
       模型预测概率: tensor([0.4334, 0.2792, 0.1553, 0.0077, 0.1244])
